# Model results — climatology · R(2+1)D · 3D-CNN · ConvLSTM · U-Net

Run this **after** training the models (`trainers/*.py`). Mirrors
`signatures/notebooks/results_cnn3d.ipynb`: it reuses each model's class + dataset
straight from its trainer (no duplication) and then

1. plots **train vs test PR-AUC over training** for every model on a single plot,
2. reloads each checkpoint and **evaluates on the validation *and* test sets**,
3. maps **ground truth vs each model's prediction** for random **validation** days.

**Layout.** Trainers live in `notebooks/model/trainers/`; all checkpoints + `*_results.npz`
(and `climatology.npz`) are written to `notebooks/model/outputs/`. Four pure-GOES
architectures (`x = (T=8, bands+lead, 1500, 2500)` → `CellPool` → per-cell logits
`(59, 95)`) plus the **climatology baseline** (per-cell training warning-rate).

## 0 · Setup

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader

# locate repo root, the model dir (gridindex + outputs), and the trainers dir
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
MODEL_DIR = ROOT / "notebooks" / "model"
TRAINER_DIR = MODEL_DIR / "trainers"
OUT_DIR = MODEL_DIR / "outputs"               # checkpoints + results npz live here
sys.path.insert(0, str(MODEL_DIR))            # for gridindex
sys.path.insert(0, str(TRAINER_DIR))          # for the trainer modules

from config import STATES_GEOJSON, build_grid_cells
from gridindex import build_pix2cell
import r2plus1d as r2p1
import cnn3d as c3d
import convlstm as clstm
import unet

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
CACHE_DIR = r2p1.CACHE_DIR

# GOES models to compare: label -> module, class, checkpoint, results npz, plot color
MODELS = {
    "R(2+1)D":  dict(m=r2p1,  cls="GOESR2plus1d", ckpt="r2plus1d.pt", res="r2plus1d_results.npz", c="#1d6fb8"),
    "3D-CNN":   dict(m=c3d,   cls="GOES3DCNN",    ckpt="cnn3d.pt",    res="cnn3d_results.npz",    c="#e09f3e"),
    "ConvLSTM": dict(m=clstm, cls="GOESConvLSTM", ckpt="convlstm.pt", res="convlstm_results.npz", c="#6a4c93"),
    "U-Net":    dict(m=unet,  cls="GOESUNet",     ckpt="unet.pt",     res="unet_results.npz",     c="#2a9d8f"),
}

# shared grid / land mask / splits (the pixel->cell index is cached by build_pix2cell)
p2c, GRID_R, GRID_C, land = build_pix2cell()
tr_days, va_days, te_days = r2p1.load_splits()


def base_rate(days):
    return float(np.mean([np.load(CACHE_DIR / f"{d}_y.npy")[land].mean() for d in days]))


BASE = {"val": base_rate(va_days), "test": base_rate(te_days)}

# climatology baseline (fitted by trainers/climatology.py) -> the bar every model must clear
clim_path = OUT_DIR / "climatology.npz"
CLIM = dict(np.load(clim_path)) if clim_path.exists() else None

# grid polygons + state outlines in Albers (EPSG:5070) for the maps
cells, _, _, _ = build_grid_cells()
cells_albers = cells.to_crs(5070)
RR, CC = cells_albers["R"].to_numpy(), cells_albers["C"].to_numpy()
states = gpd.read_file(STATES_GEOJSON)
states = states[~states["name"].isin(["Alaska", "Hawaii", "Puerto Rico"])].to_crs(5070)

print(f"device {DEVICE} | grid {GRID_R}x{GRID_C} | land {int(land.sum())} cells")
print(f"days: train {len(tr_days)}  val {len(va_days)}  test {len(te_days)}")
print(f"base flood rate: val {BASE['val']:.4f}  test {BASE['test']:.4f}")
for name, mi in MODELS.items():
    print(f"  {name:10s} checkpoint={(OUT_DIR/mi['ckpt']).exists()}  "
          f"results={(OUT_DIR/mi['res']).exists()}")
print(f"  climatology: {'loaded' if CLIM is not None else 'MISSING -> run trainers/climatology.py'}")

## 1 · (optional) Train the models

Off by default — this notebook is meant to run **after** training. Flip `TRAIN = True` to
launch from here: each GOES model trains across **both GPUs** via `torchrun`; the
climatology baseline is a quick closed-form fit. Re-running skips models whose checkpoint
already exists (unless `RETRAIN`).

In [ ]:
import subprocess

TRAIN = False        # set True to (re)train from the notebook
RETRAIN = False      # if True, retrain even when a checkpoint already exists


def _run(cmd):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run(cmd, cwd=str(MODEL_DIR), check=True)


if TRAIN:
    if CLIM is None or RETRAIN:
        _run([sys.executable, "trainers/climatology.py"])
    for name, mi in MODELS.items():
        if (OUT_DIR / mi["ckpt"]).exists() and not RETRAIN:
            print(f"[{name}] checkpoint exists - skip (set RETRAIN=True to force)")
            continue
        script = f"trainers/{Path(mi['m'].__file__).name}"
        _run([sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2", script])
    print("training done - re-run the Setup cell to refresh checkpoint status")
else:
    print("TRAIN=False - evaluating existing checkpoints. To train from a shell:")
    print("  cd notebooks/model && python trainers/climatology.py")
    for mi in MODELS.values():
        print(f"  NCCL_P2P_DISABLE=1 torchrun --nproc_per_node=2 "
              f"trainers/{Path(mi['m'].__file__).name}")

## 2 · Training curves — train vs test PR-AUC over time

One plot, all models: **solid = test PR-AUC, dashed = train PR-AUC** (same color per
model), over epochs. The gap between a model's solid and dashed line is its
generalization gap. The dotted black line is the **climatology baseline**; the grey
dashed line is the random base rate.

In [ ]:
def load_history(res):
    path = OUT_DIR / res
    if not path.exists():
        return None
    h = np.load(path)["hist"]            # cols: epoch, lr, loss, train_prauc, val_prauc, test_prauc
    return h if (h.ndim == 2 and h.shape[1] == 6) else None


fig, ax = plt.subplots(figsize=(9.5, 5.5))
for name, mi in MODELS.items():
    h = load_history(mi["res"])
    if h is None:
        print(f"{name}: no results yet - train it first")
        continue
    ep, tr, te = h[:, 0], h[:, 3], h[:, 5]
    ax.plot(ep, te, "-o", color=mi["c"], lw=2, label=f"{name} - test")
    ax.plot(ep, tr, "--", color=mi["c"], lw=1.3, alpha=0.55, label=f"{name} - train")
    print(f"{name}: best test PR-AUC {te.max():.4f} @ epoch {int(ep[te.argmax()])} "
          f"({te.max()/BASE['test']:.1f}x base)")

if CLIM is not None:
    ax.axhline(float(CLIM["test_prauc"]), ls=":", color="black", lw=1.5,
               label=f"climatology ({float(CLIM['test_prauc']):.3f})")
ax.axhline(BASE["test"], ls="--", color="0.6", lw=1, label=f"random ({BASE['test']:.3f})")
ax.set_xlabel("epoch"); ax.set_ylabel("PR-AUC (land cells)")
ax.set_title("Train vs test PR-AUC over training  (solid = test, dashed = train)")
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3 · Evaluate on validation + test

Reload each checkpoint and score it over land cells on **both** the validation and test
splits: PR-AUC (and ×base lift), ROC-AUC, and best F1 (swept over high-probability
thresholds). The climatology baseline is included as the reference row.

In [ ]:
THRESHOLD = 0.5     # default binary-map threshold (raise toward 0.8-0.9 if probs run high)


def build_model(name):
    mi = MODELS[name]; mod = mi["m"]
    sub = mod._subsample_index(p2c, mod.ENC_H, mod.ENC_W)
    net = getattr(mod, mi["cls"])(sub, GRID_R, GRID_C).to(DEVICE)
    net.load_state_dict(torch.load(OUT_DIR / mi["ckpt"], map_location=DEVICE))
    net.eval()
    return net


@torch.no_grad()
def run_inference(name, days):
    """Reload `name`'s checkpoint and predict on `days` -> probs, trues  (N, 59, 95)."""
    net = build_model(name)
    mod = MODELS[name]["m"]
    probs, trues = [], []
    for x, y in DataLoader(mod.FloodCache(days), batch_size=1, num_workers=6):
        x = x.to(DEVICE).float()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            p = torch.sigmoid(net(x)).float().squeeze(1).cpu()
        probs.append(p.numpy()[0]); trues.append(y.numpy()[0])
    del net; torch.cuda.empty_cache()
    return np.array(probs), np.array(trues)


def clim_pred(days):
    cm = CLIM["clim"]
    probs = np.broadcast_to(cm, (len(days), *cm.shape)).copy()
    trues = np.stack([np.load(CACHE_DIR / f"{d}_y.npy").astype(np.float32) for d in days])
    return probs, trues


def score(probs, trues):
    yt = trues[:, land].ravel().astype(int); p = probs[:, land].ravel()
    f1 = max(f1_score(yt, p > t, zero_division=0)
             for t in np.quantile(p, np.linspace(0.90, 0.999, 30)))
    return average_precision_score(yt, p), roc_auc_score(yt, p), f1


# run inference once per model on val + test (reused by the maps below)
have = [n for n in MODELS if (OUT_DIR / MODELS[n]["ckpt"]).exists()]
val_pred = {n: run_inference(n, va_days) for n in have}
test_pred = {n: run_inference(n, te_days) for n in have}

print(f"{'model':<12}{'split':<6}{'PR-AUC':>9}{'xbase':>7}{'ROC':>7}{'bestF1':>8}")
print("-" * 49)


def _row(name, split, probs, trues):
    ap, roc, f1 = score(probs, trues)
    print(f"{name:<12}{split:<6}{ap:>9.4f}{ap/BASE[split]:>6.1f}x{roc:>7.3f}{f1:>8.3f}")


if CLIM is not None:
    for split, days in (("val", va_days), ("test", te_days)):
        _row("climatology", split, *clim_pred(days))
for name in have:
    _row(name, "val", *val_pred[name])
    _row(name, "test", *test_pred[name])

## 4 · Ground truth vs predictions — random validation days

`N_SHOW` random **validation** days, one row each: **ground truth** then every trained
model's prediction (binarized at `THRESHOLD`, each model in its own color). Each map's
threshold-free per-day PR-AUC is printed beneath it.

In [ ]:
N_SHOW = 3          # random validation days to map
SAMPLE_SEED = 7
MODEL_CMAP = {"R(2+1)D": "Blues", "3D-CNN": "Oranges",
              "ConvLSTM": "Purples", "U-Net": "GnBu"}


def _draw(ax, values, cmap, title=None, ylabel=None, sub=None):
    gdf = cells_albers.copy(); gdf["v"] = values[RR, CC]
    gdf.boundary.plot(ax=ax, color="white", lw=0.1, zorder=2)
    gdf.plot(column="v", cmap=cmap, vmin=0, vmax=1, ax=ax, zorder=1, edgecolor="none")
    states.boundary.plot(ax=ax, color="0.55", lw=0.5, zorder=3)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11, fontweight="bold")
    if sub:
        ax.set_xlabel(sub, fontsize=9, color="0.3")


if not have:
    print("no trained checkpoints yet - train the models first (section 1)")
else:
    rng = np.random.default_rng(SAMPLE_SEED)
    sel = sorted(rng.choice(len(va_days), size=min(N_SHOW, len(va_days)), replace=False))
    ncol = 1 + len(have)
    fig, axes = plt.subplots(len(sel), ncol, figsize=(4.3 * ncol, 3.0 * len(sel)))
    if len(sel) == 1:
        axes = axes[None, :]
    for r, i in enumerate(sel):
        trues = val_pred[have[0]][1][i]
        yt = trues[land].ravel().astype(int)
        _draw(axes[r, 0], trues, "Greens",
              title="Ground truth" if r == 0 else None, ylabel=va_days[i])
        for j, name in enumerate(have, start=1):
            probs = val_pred[name][0][i]
            ap = average_precision_score(yt, probs[land].ravel())
            _draw(axes[r, j], (probs >= THRESHOLD).astype(float),
                  MODEL_CMAP.get(name, "Reds"),
                  title=name if r == 0 else None, sub=f"PR-AUC {ap:.3f}")
    fig.suptitle(f"Validation days - ground truth vs predictions (binary @ p >= {THRESHOLD:g})",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout(); plt.show()